# MEN 4338 · Week 10 — Change of support, block kriging and grade–tonnage

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/men4338/notebooks/MEN4338_W10_block_kriging_grade_tonnage.ipynb)

**IUGB · BSc Mining Engineering · Geostatistics (lecture & lab)** · Instructor: Laurent Kouadio

A mine does not dig points: it digs **blocks** (here 50 m × 50 m × 10 m), and sends each block to the mill or
to the waste dump. Block grades are averages, so they vary **less** than point grades: this is the
**support effect**. Ignoring it is a classic error: a selection made on point-like estimates promises
a tonnage that the mine never delivers.

**What you will practise:** support and dispersion variance · Krige's relation · the average variogram
$\bar\gamma$ · block kriging · grade–tonnage curves · contained metal.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/men4338/'
FILES = [('data/bassam_samples.csv', 'bassam_samples.csv'), ('data/bassam_truth_grid.csv', 'bassam_truth_grid.csv'), ('tests/w10/q1.py', 'tests/q1.py'), ('tests/w10/q2.py', 'tests/q2.py'), ('tests/w10/q3.py', 'tests/q3.py'), ('tests/w10/q4.py', 'tests/q4.py'), ('tests/w10/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

In [ ]:
# Toolbox from earlier weeks: run this cell.
import numpy as np

def spherical(h, c0, c, a):
    """Spherical model: nugget c0, partial sill c, range a. gamma(0) = 0."""
    h = np.asarray(h, float)
    r = np.minimum(h / a, 1.0)
    return np.where(h > 0, c0 + c * (1.5 * r - 0.5 * r ** 3), 0.0)

def aniso_h(dx, dy, azimuth, a_major, a_minor):
    """Distance rescaled to the major range: an anisotropic model becomes model(aniso_h(...), c0, c, a_major)."""
    t = np.radians(azimuth)
    u = dx * np.sin(t) + dy * np.cos(t)
    w = dx * np.cos(t) - dy * np.sin(t)
    return np.sqrt(u ** 2 + (w * a_major / a_minor) ** 2)

def ok_point(x0, y0, x, y, v, gamma, nmax=16):
    """Ordinary kriging at (x0, y0) with the nmax nearest data.
    gamma(dx, dy) is the variogram model, a function of the separation vector.
    Returns (estimate, kriging variance)."""
    d = np.hypot(x - x0, y - y0)
    idx = np.argsort(d)[:nmax]
    xs, ys, vs = x[idx], y[idx], v[idx]
    n = len(idx)
    A = np.ones((n + 1, n + 1)); A[n, n] = 0
    A[:n, :n] = gamma(xs[:, None] - xs[None, :], ys[:, None] - ys[None, :])
    b = np.ones(n + 1); b[:n] = gamma(xs - x0, ys - y0)
    sol = np.linalg.solve(A, b)
    w, mu = sol[:n], sol[n]
    return float(w @ vs), float(w @ b[:n] + mu)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def gamma_bassam(dx, dy):
    return spherical(aniso_h(dx, dy, 45, 300, 150), 0.06, 0.20, 300)   # total sill 0.26

samples = pd.read_csv("bassam_samples.csv")
x, y, au = samples["x"].to_numpy(), samples["y"].to_numpy(), samples["au_gpt"].to_numpy()
T = pd.read_csv("bassam_truth_grid.csv")["au_gpt"].to_numpy().reshape(100, 120)   # 10 m cells

## 1. The support effect

Average the true 10 m cells into 50 m blocks and compare the histograms: same mean, smaller spread.

In [ ]:
B = T.reshape(20, 5, 24, 5).mean(axis=(1, 3))
plt.hist(T.ravel(), bins=50, density=True, alpha=0.5, label="10 m cells")
plt.hist(B.ravel(), bins=30, density=True, alpha=0.5, label="50 m blocks")
plt.xlabel("Au (g/t)"); plt.legend(); plt.show()
print("share above 1.5 g/t:", (T >= 1.5).mean().round(3), "(cells)", (B >= 1.5).mean().round(3), "(blocks)")

## 2. The average variogram and Krige's relation

The mean of $\gamma$ over all pairs of points inside a block $v$ is $\bar\gamma(v, v)$. It is computed by
**discretising** the block into $n \times n$ points. Krige's relation links the variances of the supports:

$$D^2(v \mid D) = D^2(\cdot \mid D) - \bar\gamma(v, v)$$

For a deposit much larger than the range, $D^2(\cdot \mid D)$ is the total sill. The larger the block, the
larger $\bar\gamma(v,v)$, the smaller the variance of the blocks.

## 3. Block kriging

Same system as point kriging, but the right-hand side uses the average variogram between each sample and the
block, $\bar\gamma(\mathbf{x}_i, v)$, and the variance becomes

$$\sigma^2_{OK}(v) = \sum_i \lambda_i \bar\gamma(\mathbf{x}_i, v) + \mu - \bar\gamma(v, v)$$

## 4. Grade–tonnage curves

For each cut-off $z_c$: the tonnage of blocks with grade $\ge z_c$, and their mean grade. These two curves
drive the choice of the cut-off and the size of the mine.

---
## Exercises

### Q1. Point and block variances

Store the true 50 m blocks in `blocks_true` (shape 20 × 24), the variance of the 10 m cells in `var_points`,
that of the blocks in `var_blocks`, and their ratio in `f_ratio`.

In [ ]:
blocks_true = ...
var_points = ...
var_blocks = ...
f_ratio = ...
var_points, var_blocks, f_ratio

In [ ]:
grader.check("q1")

### Q2. Discretisation and $\bar\gamma$

Write `discretise(xc, yc, size, n)` returning the arrays `(px, py)` of the $n \times n$ points at the centres
of the sub-cells of a square block of side `size` centred on `(xc, yc)`. Then write
`gamma_bar(gamma, size, n=5)`, the mean of `gamma` over all pairs of these points. Store
`gbar_50 = gamma_bar(gamma_bassam, 50, 5)` and the dispersion variance of 50 m blocks predicted by the model,
`block_var_model` = 0.26 − `gbar_50`.

In [ ]:
def discretise(xc, yc, size, n):
    ...

def gamma_bar(gamma, size, n=5):
    ...

gbar_50 = ...
block_var_model = ...
gbar_50, block_var_model

In [ ]:
grader.check("q2")

### Q3. Block kriging

Write `ok_block(xc, yc, size, x, y, v, gamma, nd=4, nmax=16)` returning `(estimate, variance)` for the block,
using the `nmax` samples closest to the block centre and $nd \times nd$ discretisation points.

In [ ]:
def ok_block(xc, yc, size, x, y, v, gamma, nd=4, nmax=16):
    ...

ok_block(625., 225., 50, x, y, au, gamma_bassam), ok_point(625., 225., x, y, au, gamma_bassam)

In [ ]:
grader.check("q3")

### Q4. The block model

Krig every 50 m block (centre `(25 + 50*c, 25 + 50*r)`): `bk_map` and `bk_var`, shape 20 × 24. Store the mean
squared error against the true blocks in `bk_mse`.

In [ ]:
bk_map = np.zeros((20, 24))
bk_var = np.zeros((20, 24))
...
bk_mse = ...
bk_mse

In [ ]:
grader.check("q4")

### Q5. Grade–tonnage

With `cutoffs = np.arange(0, 2.01, 0.25)` and 67 500 t per block, compute for the kriged blocks the tonnage
above each cut-off in Mt (`tonnes_mt`) and the mean grade above (`grade_above`), and the true tonnage curve
`tonnes_true_mt`. Store the gold contained in the blocks estimated ≥ 1.0 g/t in thousands of ounces
(`metal_koz`). Plot the curves. At which cut-offs does the kriged block model over- or under-state tonnage?

In [ ]:
cutoffs = ...
tonnes_mt = ...
grade_above = ...
tonnes_true_mt = ...
metal_koz = ...
metal_koz

In [ ]:
grader.check("q5")

### Going further (not checked)

* Compute the tonnage above 1.5 g/t from the **point** kriging map of week 9 (20 m nodes) as if each node were a
  block. Why is it different from the 50 m block result?
* Compute `gamma_bar` for blocks of 10, 25, 50, 100 m and plot the block dispersion variance against block size.

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()